In [1]:
import os
import re
from typing import TypedDict, List, Annotated, Literal
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
import json
from datetime import datetime

load_dotenv()

c:\Users\Edunet Foundation\anaconda3\envs\agent-h\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
import os
from typing import TypedDict, List, Annotated, Literal
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
import json
from datetime import datetime

load_dotenv()

True

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [4]:
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
SERPAPI_API_KEY = os.getenv("SERPAPI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
EXA_API_KEY = os.getenv("EXA_API_KEY")

if not DEEPSEEK_API_KEY:
    raise ValueError("Please set DEEPSEEK_API_KEY in your environment variables")
if not SERPAPI_API_KEY:
    raise ValueError("Please set SERPAPI_API_KEY in your environment variables")
if not TAVILY_API_KEY:
    raise ValueError("Please set TAVILY_API_KEY in your environment variables")
if not EXA_API_KEY:
    raise ValueError("Please set EXA_API_KEY in your environment variables")

In [5]:
# ==================== TOOL 1: GOOGLE TRENDS (SerpAPI) ====================

@tool
def google_trends_tool(startup_idea: str) -> str:
    """
    Get Google Trends data for a startup idea.
    Returns trend score (0-100) and related queries.
    """
    if not SERPAPI_API_KEY:
        return json.dumps({
            "trend_score": 50,
            "trend_data": "SerpAPI key not configured",
            "error": "SERPAPI_API_KEY not set"
        })
    
    try:
        from serpapi import GoogleSearch
        
        # Extract keywords from idea
        keywords = startup_idea[:100].split()[:5]
        query = " ".join(keywords)
        
        params = {
            "api_key": SERPAPI_API_KEY,
            "engine": "google_trends",
            "q": query,
            "data_type": "TIMESERIES"
        }
        
        search = GoogleSearch(params)
        results = search.get_dict()
        
        # Extract interest over time
        interest_over_time = results.get("interest_over_time", {})
        timeline_data = interest_over_time.get("timeline_data", [])
        
        if timeline_data:
            # Calculate average interest score
            scores = [point.get("value", [{}])[0].get("value", 0) for point in timeline_data if point.get("value")]
            avg_score = sum(scores) / len(scores) if scores else 50
        else:
            avg_score = 50
        
        return json.dumps({
            "trend_score": int(avg_score),
            "query": query,
            "data_points": len(timeline_data),
            "source": "Google Trends via SerpAPI"
        })
    except Exception as e:
        return json.dumps({
            "trend_score": 50,
            "error": str(e),
            "note": "Using default score"
        })

# ==================== TOOL 2: COMPETITOR SEARCH (Tavily) ====================

@tool
def competitor_search_tool(startup_idea: str) -> str:
    """
    Search for competitors using Tavily API.
    Returns list of competitors with descriptions.
    """
    if not TAVILY_API_KEY:
        return json.dumps({
            "competitors": [],
            "error": "TAVILY_API_KEY not set",
            "note": "Get key from https://tavily.com"
        })
    
    try:
        from tavily import TavilyClient
        
        client = TavilyClient(api_key=TAVILY_API_KEY)
        
        search_query = f"top competitors for {startup_idea[:150]} startup company"
        response = client.search(
            query=search_query,
            search_depth="advanced",
            max_results=8,
            include_answer=True
        )
        
        competitors = []
        for result in response.get('results', []):
            competitors.append({
                "name": result.get('title', 'Unknown')[:50],
                "description": result.get('content', '')[:200],
                "url": result.get('url', ''),
                "relevance": result.get('score', 0)
            })
        
        return json.dumps({
            "competitors": competitors,
            "summary": response.get('answer', '')[:500],
            "total_found": len(competitors),
            "source": "Tavily API"
        })
    except Exception as e:
        return json.dumps({
            "competitors": [],
            "error": str(e)
        })

# ==================== TOOL 3: MARKET VALIDATION (Exa AI) ====================

@tool
def market_validation_tool(startup_idea: str) -> str:
    """
    Search for similar products and market trends using Exa AI.
    Returns traction signals and market interest.
    """
    if not EXA_API_KEY:
        return json.dumps({
            "similar_products": [],
            "error": "EXA_API_KEY not set",
            "note": "Get key from https://exa.ai"
        })
    
    try:
        from exa_py import Exa
        
        exa = Exa(api_key=EXA_API_KEY)
        
        # Search for similar products
        results = exa.search(
            f"startup product similar to {startup_idea[:100]} launch funding",
            type="auto",
            num_results=6,
            contents={"highlights": True}
        )
        
        similar_products = []
        for result in results.results:
            similar_products.append({
                "title": result.title[:100] if result.title else "Unknown",
                "url": result.url if result.url else "",
                "highlights": result.highlights[:200] if hasattr(result, 'highlights') and result.highlights else ""
            })
        
        # Get market trends
        trends = exa.search(
            f"{startup_idea[:80]} market trends 2025 2026",
            type="auto",
            num_results=4,
            contents={"highlights": True}
        )
        
        market_trends = []
        for t in trends.results[:3]:
            market_trends.append({
                "title": t.title[:100] if t.title else "Unknown",
                "url": t.url if t.url else ""
            })
        
        return json.dumps({
            "similar_products": similar_products,
            "market_trends": market_trends,
            "similar_count": len(similar_products),
            "source": "Exa AI"
        })
    except Exception as e:
        return json.dumps({
            "similar_products": [],
            "error": str(e)
        })

# ==================== TOOL 4: IDEA SCORER (Combines all data) ====================

@tool
def idea_scorer_tool(
    startup_idea: str,
    trend_score: int,
    competitors: list,
    market_validation: dict
) -> str:
    """
    Calculate final startup score (0-100) combining all data sources.
    Weighted by: Market Trend (30%), Competition (25%), Validation (25%), Innovation (20%)
    """
    # Market Trend Score (0-100)
    trend_weighted = trend_score * 0.30
    
    # Competition Score (fewer competitors = higher score, but some competition validates market)
    num_competitors = len(competitors) if isinstance(competitors, list) else 0
    if num_competitors == 0:
        competition_score = 60  # No competitors - could be blue ocean or no market
    elif num_competitors <= 3:
        competition_score = 80  # Healthy competition, market validated
    elif num_competitors <= 6:
        competition_score = 60  # Moderate competition
    elif num_competitors <= 10:
        competition_score = 40  # Crowded
    else:
        competition_score = 25  # Saturated
    competition_weighted = competition_score * 0.25
    
    # Market Validation Score (based on similar products)
    similar_count = market_validation.get('similar_count', 0) if isinstance(market_validation, dict) else 0
    if similar_count == 0:
        validation_score = 50  # No direct competition found
    elif similar_count <= 2:
        validation_score = 75  # Some validation, not overcrowded
    elif similar_count <= 5:
        validation_score = 60  # Established market
    else:
        validation_score = 40  # Many similar products
    validation_weighted = validation_score * 0.25
    
    # Innovation Score - based on idea uniqueness (LLM evaluates)
    # This will be calculated by the main agent
    innovation_score = 65  # Default, will be overridden
    innovation_weighted = innovation_score * 0.20
    
    final_score = trend_weighted + competition_weighted + validation_weighted + innovation_weighted
    
    # Determine recommendation
    if final_score >= 75:
        recommendation = "HIGH POTENTIAL - Strong market signals, proceed with MVP"
        urgency = "Move quickly to capture market opportunity"
    elif final_score >= 55:
        recommendation = "MEDIUM POTENTIAL - Refine positioning and validate further"
        urgency = "Conduct customer discovery interviews"
    elif final_score >= 35:
        recommendation = "LOW POTENTIAL - Significant pivot or repositioning needed"
        urgency = "Re-evaluate problem-solution fit"
    else:
        recommendation = "UNFAVORABLE - Consider different direction or market"
        urgency = "Pivot or explore adjacent markets"
    
    return json.dumps({
        "final_score": round(final_score, 1),
        "breakdown": {
            "market_trend": round(trend_weighted, 1),
            "competition": round(competition_weighted, 1),
            "market_validation": round(validation_weighted, 1),
            "innovation": round(innovation_weighted, 1)
        },
        "raw_scores": {
            "trend_score": trend_score,
            "competition_score": competition_score,
            "validation_score": validation_score,
            "innovation_score": innovation_score
        },
        "recommendation": recommendation,
        "urgency": urgency,
        "weights": {
            "market_trend": "30%",
            "competition": "25%",
            "market_validation": "25%",
            "innovation": "20%"
        }
    })


In [6]:
# ==================== TRADITIONAL TOOLS ====================

@tool
def calculate_tax_liability(revenue: float, expenses: float, jurisdiction: str) -> str:
    """Calculate tax liability based on revenue, expenses, and jurisdiction."""
    taxable_income = revenue - expenses
    tax_rates = {
        "USA": 0.21,
        "UK": 0.19,
        "INDIA": 0.25,
        "GERMANY": 0.30
    }
    rate = tax_rates.get(jurisdiction.upper(), 0.25)
    tax = taxable_income * rate
    return json.dumps({
        "tax_due": round(tax, 2),
        "taxable_income": round(taxable_income, 2),
        "effective_rate": rate,
        "currency": "USD" if jurisdiction == "USA" else "Local"
    })

@tool
def generate_payslip(employee_name: str, salary: float, deductions: float, month: str) -> str:
    """Generate a payslip for an employee."""
    net_pay = salary - deductions
    return json.dumps({
        "employee": employee_name,
        "month": month,
        "gross_salary": salary,
        "deductions": deductions,
        "net_pay": net_pay,
        "generated_date": datetime.now().isoformat()
    })

@tool
def check_domain_availability(domain_name: str, tld: str = "com") -> str:
    """Check if a domain name is available for purchase."""
    unavailable = ["google", "facebook", "amazon", "microsoft"]
    if domain_name.lower() in unavailable:
        return json.dumps({"available": False, "domain": f"{domain_name}.{tld}", "suggestion": f"Try {domain_name}ai.{tld}"})
    return json.dumps({
        "available": True,
        "domain": f"{domain_name}.{tld}",
        "price": 12.99,
        "registrar": "GoDaddy",
        "purchase_url": f"https://example.com/buy/{domain_name}"
    })

@tool
def add_to_crm(lead_name: str, email: str, stage: str, score: int) -> str:
    """Add a lead to the CRM system."""
    return json.dumps({
        "status": "success",
        "lead_id": f"LEAD_{datetime.now().timestamp()}",
        "name": lead_name,
        "email": email,
        "stage": stage,
        "score": score,
        "added_at": datetime.now().isoformat()
    })

@tool
def calculate_vc_valuation(revenue: float, growth_rate: float, industry_multiplier: float = 10) -> str:
    """Calculate startup valuation for VC funding."""
    valuation = revenue * industry_multiplier * (1 + growth_rate/100)
    return json.dumps({
        "valuation": round(valuation, 2),
        "method": "Revenue Multiple",
        "multiple_used": industry_multiplier,
        "adjusted_for_growth": growth_rate,
        "recommended_raise": round(valuation * 0.20, 2)
    })


In [7]:
# ==================== STATE DEFINITION ====================

class FounderState(TypedDict):
    messages: List[Annotated[HumanMessage | AIMessage, "Conversation history"]]
    current_task: str
    idea_description: str
    idea_score: int
    market_research_results: str
    financial_metrics: dict
    domain_info: dict
    tax_info: dict
    vc_info: dict
    ca_info: dict
    crm_status: str
    payslip_info: dict
    astrology_advice: str
    # New fields for real data
    trend_score: int
    competitors_data: dict
    market_validation_data: dict
    quantitative_score: float

In [8]:
# ==================== INITIALIZE LLMs ====================

llm_pro = ChatDeepSeek(
    model="deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    temperature=0.3,
    max_tokens=4096
)

llm_flash = ChatDeepSeek(
    model="deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    temperature=0.9,
    max_tokens=2048
)

In [9]:
# ==================== ENHANCED AGENT 1: IDEA EVALUATION (with real data) ====================

def idea_evaluation_agent(state: FounderState):
    """Evaluates startup idea using real data from Google Trends, Tavily, and Exa AI"""
    
    idea = state.get("idea_description", "")
    
    print("\n📊 Collecting real-time market data...")
    
    # Step 1: Get Google Trends data
    print("   🔍 Fetching Google Trends data...")
    trend_result = google_trends_tool.invoke({"startup_idea": idea})
    trend_data = json.loads(trend_result)
    trend_score = trend_data.get("trend_score", 50)
    print(f"      📈 Trend Score: {trend_score}/100")
    
    # Step 2: Find competitors
    print("   🔍 Searching for competitors...")
    competitor_result = competitor_search_tool.invoke({"startup_idea": idea})
    competitor_data = json.loads(competitor_result)
    competitors = competitor_data.get("competitors", [])
    print(f"      🏢 Found {len(competitors)} competitors")
    
    # Step 3: Market validation
    print("   🔍 Checking market validation...")
    validation_result = market_validation_tool.invoke({"startup_idea": idea})
    validation_data = json.loads(validation_result)
    print(f"      ✅ Found {validation_data.get('similar_count', 0)} similar products")
    
    # Step 4: Calculate quantitative score
    print("   🔍 Calculating quantitative score...")
    
    # Evaluate innovation using LLM
    innovation_prompt = f"""Rate the innovation level of this startup idea from 0-100:
    
Idea: {idea}

Consider:
- Uniqueness (is this truly novel or a copy?)
- Technology advancement
- Business model innovation
- Barrier to entry

Respond ONLY with a number between 0-100."""
    
    innovation_response = llm_flash.invoke([
        SystemMessage(content="You are an innovation expert. Respond with only a number."),
        HumanMessage(content=innovation_prompt)
    ])
    
    try:
        innovation_score = int(re.search(r'\d+', innovation_response.content).group())
        innovation_score = min(100, max(0, innovation_score))
    except:
        innovation_score = 65
    
    # Update validation data with innovation score
    validation_data['innovation_score'] = innovation_score
    
    # Calculate final score
    score_result = idea_scorer_tool.invoke({
        "startup_idea": idea,
        "trend_score": trend_score,
        "competitors": competitors,
        "market_validation": validation_data
    })
    score_data = json.loads(score_result)
    
    # Step 5: Generate detailed evaluation report with LLM
    print("   📝 Generating comprehensive report...")
    
    evaluation_prompt = f"""Based on REAL MARKET DATA, evaluate this startup idea:

STARTUP IDEA: {idea}

QUANTITATIVE DATA:
- Google Trends Score: {trend_score}/100
- Competitors Found: {len(competitors)}
- Similar Products Found: {validation_data.get('similar_count', 0)}
- Innovation Score: {innovation_score}/100
- Final Score: {score_data['final_score']}/100
- Recommendation: {score_data['recommendation']}

COMPETITOR LIST:
{json.dumps(competitors[:3], indent=2) if competitors else 'No competitors found'}

MARKET TRENDS:
{json.dumps(validation_data.get('market_trends', [])[:2], indent=2)}

Provide a detailed evaluation with:
1. **EXECUTIVE SUMMARY** (2-3 sentences)
2. **SCORE BREAKDOWN** (explain each component)
3. **MARKET OPPORTUNITY** (problems, gaps, your advantage)
4. **COMPETITIVE LANDSCAPE** (threats and differentiation)
5. **TOP 3 RISKS** (specific and actionable)
6. **TOP 3 OPPORTUNITIES** (how to win)
7. **ACTIONABLE RECOMMENDATIONS** (next steps)

Be brutally honest and data-driven."""
    
    response = llm_pro.invoke([
        SystemMessage(content="You are a senior startup analyst with VC experience."),
        HumanMessage(content=evaluation_prompt)
    ])
    
    # Format final score display
    final_output = f"""
{'='*60}
📊 QUANTITATIVE SCORE BREAKDOWN
{'='*60}

🎯 FINAL SCORE: {score_data['final_score']}/100
📈 RECOMMENDATION: {score_data['recommendation']}
⚡ URGENCY: {score_data['urgency']}

Detailed Breakdown:
   📊 Market Trend:     {score_data['breakdown']['market_trend']}/30
   🏢 Competition:      {score_data['breakdown']['competition']}/25
   ✅ Validation:       {score_data['breakdown']['market_validation']}/25
   💡 Innovation:       {score_data['breakdown']['innovation']}/20

Raw Scores:
   Google Trends:       {trend_score}/100
   Competition Level:   {score_data['raw_scores']['competition_score']}/100
   Market Validation:   {score_data['raw_scores']['validation_score']}/100
   Innovation:          {innovation_score}/100

{'='*60}
📋 DETAILED ANALYSIS
{'='*60}

{response.content}
"""
    
    return {
        "messages": [AIMessage(content=final_output)],
        "idea_score": int(score_data['final_score'] / 10),  # Convert to 1-10 scale
        "trend_score": trend_score,
        "competitors_data": competitor_data,
        "market_validation_data": validation_data,
        "quantitative_score": score_data['final_score'],
        "current_task": "evaluated"
    }

# ==================== AGENT 2: MARKET RESEARCH (Enhanced) ====================

def market_research_agent(state: FounderState):
    """Conducts thorough market research with problems and solutions"""
    idea = state.get("idea_description", "")
    quantitative_score = state.get("quantitative_score", 0)
    competitors = state.get("competitors_data", {}).get("competitors", [])
    
    system_prompt = f"""You are a senior market research analyst. Using this data:

QUANTITATIVE SCORE: {quantitative_score}/100
COMPETITORS FOUND: {len(competitors)}

Provide a COMPREHENSIVE market research report with:

**1. CURRENT MARKET PROBLEMS (3-5 specific pain points)**
List major problems customers face. Be specific and actionable.

**2. EXISTING SOLUTIONS & COMPETITORS**
Based on the competitors found: {json.dumps([c.get('name') for c in competitors[:3]]) if competitors else 'No specific competitors found'}
What are their approaches and limitations?

**3. MARKET GAPS (What's missing)**
What problems remain unsolved? Where are competitors falling short?

**4. YOUR UNIQUE OPPORTUNITY**
How can this specific idea fill the gaps?

**5. MARKET SIZE & TRENDS**
Estimated TAM, SAM, SOM. Growth rates and emerging trends.

Be data-driven, specific, and brutally honest."""
    
    response = llm_pro.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Startup Idea: {idea}")
    ])
    
    return {
        "messages": [response],
        "market_research_results": response.content,
        "current_task": "market_researched"
    }

# ==================== AGENT 3: DOMAIN PURCHASE ====================

def domain_purchase_agent(state: FounderState):
    """Checks and suggests domain names"""
    idea = state.get("idea_description", "")
    
    response = llm_flash.invoke([
        SystemMessage(content="Generate 5 potential domain names based on this startup idea. Keep them short and memorable."),
        HumanMessage(content=idea)
    ])
    
    words = re.findall(r'\b\w+\b', response.content[:200])
    suggested_name = words[0] if words else "startup"
    
    availability = check_domain_availability.invoke({"domain_name": suggested_name})
    
    return {
        "messages": [response, AIMessage(content=f"Domain check: {availability}")],
        "domain_info": json.loads(availability),
        "current_task": "domain_checked"
    }

# ==================== AGENT 4: TAX FILING ====================

def tax_filing_agent(state: FounderState):
    """Handles tax calculations and filings"""
    financials = state.get("financial_metrics", {"revenue": 0, "expenses": 0})
    
    if financials.get("revenue", 0) == 0:
        return {
            "messages": [AIMessage(content="No financial data provided. Tax calculation skipped.")],
            "tax_info": {},
            "current_task": "tax_skipped"
        }
    
    tax_result = calculate_tax_liability.invoke({
        "revenue": financials.get("revenue", 0),
        "expenses": financials.get("expenses", 0),
        "jurisdiction": "USA"
    })
    
    response = llm_pro.invoke([
        SystemMessage(content="Explain this tax calculation in simple terms."),
        HumanMessage(content=f"Tax calculation result: {tax_result}")
    ])
    
    return {
        "messages": [response],
        "tax_info": json.loads(tax_result),
        "current_task": "tax_calculated"
    }

# ==================== AGENT 5: VC FUNDING ====================

def vc_funding_agent(state: FounderState):
    """Prepares for VC funding and valuations"""
    financials = state.get("financial_metrics", {"revenue": 0, "growth_rate": 0})
    quantitative_score = state.get("quantitative_score", 0)
    
    if financials.get("revenue", 0) == 0:
        return {
            "messages": [AIMessage(content="No revenue data provided. VC valuation skipped.")],
            "vc_info": {},
            "current_task": "vc_skipped"
        }
    
    valuation = calculate_vc_valuation.invoke({
        "revenue": financials.get("revenue", 0),
        "growth_rate": financials.get("growth_rate", 50),
        "industry_multiplier": 10
    })
    
    response = llm_pro.invoke([
        SystemMessage(content=f"You are a VC partner. The startup scored {quantitative_score}/100 on our evaluation. Provide funding advice."),
        HumanMessage(content=f"Valuation: {valuation}\nIdea: {state.get('idea_description', '')[:200]}")
    ])
    
    return {
        "messages": [response],
        "vc_info": json.loads(valuation),
        "current_task": "vc_advised"
    }

# ==================== AGENT 6: CA WORK ====================

def ca_work_agent(state: FounderState):
    """Handles accounting, bookkeeping, and compliance"""
    response = llm_pro.invoke([
        SystemMessage(content="""You are a Chartered Accountant. Provide guidance on:
        1. Required registrations
        2. Bookkeeping best practices
        3. Expense tracking
        4. Compliance calendar"""),
        HumanMessage(content=f"Idea: {state.get('idea_description', '')[:200]}")
    ])
    
    return {
        "messages": [response],
        "ca_info": {"advice": response.content[:500]},
        "current_task": "ca_advised"
    }

# ==================== AGENT 7: PAYSLIP GENERATOR ====================

def payslip_generator_agent(state: FounderState):
    """Generates payslips for employees"""
    payslip = generate_payslip.invoke({
        "employee_name": "Sample Employee",
        "salary": 75000,
        "deductions": 15000,
        "month": datetime.now().strftime("%B %Y")
    })
    
    response = llm_flash.invoke([
        SystemMessage(content="Generate a professional email to send this payslip."),
        HumanMessage(content=f"Payslip data: {payslip}")
    ])
    
    return {
        "messages": [response],
        "payslip_info": json.loads(payslip),
        "current_task": "payslip_generated"
    }

# ==================== AGENT 8: ASTROLOGY SUPPORT ====================

def astrology_support_agent(state: FounderState):
    """Provides motivational astrology guidance"""
    idea = state.get("idea_description", "")
    score = state.get("idea_score", 5)
    quantitative_score = state.get("quantitative_score", 0)
    
    response = llm_flash.invoke([
        SystemMessage(content="""Provide motivational astrology guidance:
        1. Zodiac reading for the founder based on their startup journey
        2. Auspicious launch date in next 30 days
        3. Motivational message tied to their score"""),
        HumanMessage(content=f"Startup: {idea}\nScore: {score}/10 (Quantitative: {quantitative_score}/100)")
    ])
    
    return {
        "messages": [response],
        "astrology_advice": response.content,
        "current_task": "astrology_done"
    }

# ==================== AGENT 9: CRM INTEGRATION ====================

def crm_integration_agent(state: FounderState):
    """Integrates with CRM system"""
    score = state.get("idea_score", 0)
    quantitative_score = state.get("quantitative_score", 0)
    
    crm_result = add_to_crm.invoke({
        "lead_name": f"Startup_{datetime.now().strftime('%Y%m%d')}",
        "email": "founder@example.com",
        "stage": "Idea Validation",
        "score": int(quantitative_score)
    })
    
    response = llm_pro.invoke([
        SystemMessage(content="Summarize CRM addition and suggest next steps based on the quantitative score."),
        HumanMessage(content=f"CRM result: {crm_result}\nScore: {quantitative_score}/100")
    ])
    
    return {
        "messages": [response],
        "crm_status": crm_result,
        "current_task": "crm_updated"
    }

In [10]:
# ==================== BUILD THE GRAPH ====================

def build_founder_assistant():
    """Build and compile the complete agent graph - ALL agents run sequentially"""
    
    workflow = StateGraph(FounderState)
    
    # Add all 9 agents
    workflow.add_node("idea_evaluation_agent", idea_evaluation_agent)
    workflow.add_node("market_research_agent", market_research_agent)
    workflow.add_node("domain_purchase_agent", domain_purchase_agent)
    workflow.add_node("tax_filing_agent", tax_filing_agent)
    workflow.add_node("vc_funding_agent", vc_funding_agent)
    workflow.add_node("ca_work_agent", ca_work_agent)
    workflow.add_node("payslip_generator_agent", payslip_generator_agent)
    workflow.add_node("astrology_support_agent", astrology_support_agent)
    workflow.add_node("crm_integration_agent", crm_integration_agent)
    
    # Set entry point
    workflow.set_entry_point("idea_evaluation_agent")
    
    # Sequential flow - ALL agents run
    workflow.add_edge("idea_evaluation_agent", "market_research_agent")
    workflow.add_edge("market_research_agent", "domain_purchase_agent")
    workflow.add_edge("domain_purchase_agent", "vc_funding_agent")
    workflow.add_edge("vc_funding_agent", "ca_work_agent")
    workflow.add_edge("ca_work_agent", "tax_filing_agent")
    workflow.add_edge("tax_filing_agent", "payslip_generator_agent")
    workflow.add_edge("payslip_generator_agent", "astrology_support_agent")
    workflow.add_edge("astrology_support_agent", "crm_integration_agent")
    workflow.add_edge("crm_integration_agent", END)
    
    return workflow.compile()


In [11]:
# ==================== DISPLAY FUNCTIONS ====================

def display_welcome():
    """Display welcome banner"""
    print("\n" + "=" * 70)
    print("🚀 FOUNDER'S AI ASSISTANT (Powered by Real Data)")
    print("=" * 70)
    print("\nData Sources:")
    print("  📊 Google Trends (SerpAPI) - Real market interest")
    print("  🔍 Tavily - Live competitor discovery")
    print("  🎯 Exa AI - Product Hunt & market validation")
    print("  🤖 DeepSeek V4 - Analysis & recommendations")
    print("\nI analyze your startup idea and provide:")
    print("  📊 1. Idea Evaluation (Data-driven score 0-100)")
    print("  📈 2. Market Research (Problems + Solutions)")
    print("  🌐 3. Domain Suggestions")
    print("  💰 4. VC Funding Analysis")
    print("  📋 5. CA/Compliance Advice")
    print("  💸 6. Tax Calculation")
    print("  📄 7. Sample Payslip")
    print("  ✨ 8. Astrology Motivation")
    print("  🤝 9. CRM Integration")
    print("\n" + "=" * 70)

def display_report_menu():
    """Show report selection menu"""
    print("\n" + "=" * 70)
    print("📋 REPORT SELECTION")
    print("=" * 70)
    print("Which reports would you like to see?")
    print("  1. Market Research (Problems & Solutions) - RECOMMENDED")
    print("  2. Full Analysis (All reports)")
    print("  3. Custom selection")
    print("  4. Summary only (Score + key insights)")
    
    choice = input("\n👉 Enter your choice (1-4): ").strip()
    return choice

def display_custom_menu():
    """Display custom report selection"""
    print("\nSelect reports to display (comma-separated, e.g., 1,2,3):")
    print("  1. Idea Evaluation (Data-driven score)")
    print("  2. Market Research (Problems & Solutions)")
    print("  3. Domain Information")
    print("  4. VC Funding")
    print("  5. CA Advice")
    print("  6. Tax Calculation")
    print("  7. Payslip")
    print("  8. Astrology")
    print("  9. CRM Status")
    
    choice = input("\n👉 Your selection: ").strip()
    selected = [int(x.strip()) for x in choice.split(",") if x.strip().isdigit()]
    return selected

def display_report(final_state, selected_reports):
    """Display selected reports"""
    reports = {
        1: ("📊 DATA-DRIVEN EVALUATION", final_state.get("quantitative_score"), 
            final_state["messages"][0].content if final_state["messages"] else None),
        2: ("📈 MARKET RESEARCH (Problems & Solutions)", final_state.get("market_research_results"), 
            final_state.get("market_research_results")),
        3: ("🌐 DOMAIN INFORMATION", final_state.get("domain_info"), None),
        4: ("💰 VC FUNDING", final_state.get("vc_info"), None),
        5: ("📋 CA ADVICE", final_state.get("ca_info"), final_state.get("ca_info", {}).get("advice")),
        6: ("💸 TAX CALCULATION", final_state.get("tax_info"), None),
        7: ("📄 PAYSLIP", final_state.get("payslip_info"), None),
        8: ("✨ ASTROLOGY INSIGHTS", final_state.get("astrology_advice"), final_state.get("astrology_advice")),
        9: ("🤝 CRM STATUS", final_state.get("crm_status"), final_state.get("crm_status"))
    }
    
    for num in sorted(selected_reports):
        if num in reports:
            title, data, content = reports[num]
            print("\n" + "=" * 70)
            print(title)
            print("=" * 70)
            
            if num == 1:
                print(f"\n🎯 Quantitative Score: {final_state.get('quantitative_score', 'N/A')}/100")
                print(f"📊 Converted Score: {final_state.get('idea_score', 'N/A')}/10")
                if content:
                    print("\n" + content)
                
            elif num == 2 and content:
                print(content)
                
            elif num == 3 and data:
                print(f"\n  Domain: {data.get('domain', 'N/A')}")
                print(f"  Available: {'✅ YES' if data.get('available') else '❌ NO'}")
                if data.get('price'):
                    print(f"  Price: ${data['price']}")
                    
            elif num == 4 and data:
                print(f"\n  Valuation: ${data.get('valuation', 'N/A')}")
                print(f"  Method: {data.get('method', 'N/A')}")
                print(f"  Recommended Raise: ${data.get('recommended_raise', 'N/A')}")
                
            elif num == 5 and content:
                print(f"\n{content}")
                
            elif num == 6 and data:
                print(f"\n  Tax Due: ${data.get('tax_due', 'N/A')}")
                print(f"  Taxable Income: ${data.get('taxable_income', 'N/A')}")
                print(f"  Effective Rate: {data.get('effective_rate', 0) * 100}%")
                
            elif num == 7 and data:
                print(f"\n  Employee: {data.get('employee', 'N/A')}")
                print(f"  Net Pay: ${data.get('net_pay', 'N/A')}")
                print(f"  Month: {data.get('month', 'N/A')}")
                
            elif num == 8 and content:
                print(f"\n{content[:600]}")
                
            elif num == 9 and content:
                print(f"\n{content[:200]}")

In [12]:
# ==================== MAIN EXECUTION ====================

def main():
    """Main interactive execution"""
    display_welcome()
    
    # Check API keys
    print("\n🔧 API Status:")
    print(f"   DeepSeek: {'✅' if DEEPSEEK_API_KEY else '❌'}")
    print(f"   SerpAPI:  {'✅' if SERPAPI_API_KEY else '❌'} (Google Trends)")
    print(f"   Tavily:   {'✅' if TAVILY_API_KEY else '❌'} (Competitors)")
    print(f"   Exa AI:   {'✅' if EXA_API_KEY else '❌'} (Market Validation)")
    
    if not any([SERPAPI_API_KEY, TAVILY_API_KEY, EXA_API_KEY]):
        print("\n⚠️ WARNING: No external API keys found!")
        print("   The evaluation will use mock data.")
        print("   Sign up for free accounts at:")
        print("   - SerpAPI: https://serpapi.com (Google Trends)")
        print("   - Tavily: https://tavily.com (Competitors)")
        print("   - Exa AI: https://exa.ai (Market validation)")
    
    # Get startup idea
    print("\n💡 TELL ME ABOUT YOUR STARTUP IDEA")
    print("   (Include: problem you solve, target customer, what's unique)\n")
    idea_description = input("👉 Your idea: ").strip()
    
    if not idea_description:
        print("\n❌ No idea provided. Exiting.")
        return
    
    print(f"\n✅ Analyzing: \"{idea_description[:100]}...\"")
    
    # Optional financial metrics
    print("\n💰 FINANCIAL METRICS (Optional - improves accuracy)")
    print("   Press Enter to skip any question\n")
    
    financial_metrics = {"revenue": 0, "expenses": 0, "growth_rate": 0}
    
    try:
        rev = input("   Monthly revenue (USD): $").strip()
        if rev:
            financial_metrics["revenue"] = float(rev)
        
        exp = input("   Monthly expenses (USD): $").strip()
        if exp:
            financial_metrics["expenses"] = float(exp)
        
        growth = input("   Monthly growth rate (%): ").strip()
        if growth:
            financial_metrics["growth_rate"] = float(growth)
    except ValueError:
        print("   ⚠️ Invalid number. Using defaults.")
    
    # Create assistant and initial state
    assistant = build_founder_assistant()
    
    initial_state = {
        "messages": [],
        "current_task": "start",
        "idea_description": idea_description,
        "idea_score": 0,
        "market_research_results": "",
        "financial_metrics": financial_metrics,
        "domain_info": {},
        "tax_info": {},
        "vc_info": {},
        "ca_info": {},
        "crm_status": "",
        "payslip_info": {},
        "astrology_advice": "",
        "trend_score": 0,
        "competitors_data": {},
        "market_validation_data": {},
        "quantitative_score": 0
    }
    
    print("\n" + "=" * 70)
    print("🤖 ANALYZING YOUR STARTUP WITH REAL DATA...")
    print("   Running all 9 agents (takes 30-45 seconds)\n")
    
    # Show progress
    agents = ["Idea Evaluation (with real data)", "Market Research", "Domain Check", "VC Analysis", 
              "CA Advice", "Tax Calculation", "Payslip", "Astrology", "CRM"]
    
    for i, agent in enumerate(agents, 1):
        print(f"   🔄 [{i}/9] {agent}...", end=" ", flush=True)
    
    print("\n")
    
    # Run the workflow
    final_state = assistant.invoke(initial_state)
    
    print("\n✅ Analysis complete!\n")
    
    # Ask for report preference
    menu_choice = display_report_menu()
    
    if menu_choice == "1":
        # Market research only
        if final_state.get("market_research_results"):
            print("\n" + "=" * 70)
            print("📈 MARKET RESEARCH: PROBLEMS & SOLUTIONS")
            print("=" * 70)
            print(final_state["market_research_results"])
        else:
            print("\n❌ Market research data not available.")
            
    elif menu_choice == "2":
        # Full analysis
        display_report(final_state, range(1, 10))
        
    elif menu_choice == "3":
        # Custom selection
        selected = display_custom_menu()
        display_report(final_state, selected)
        
    else:
        # Summary only
        print("\n" + "=" * 70)
        print("📊 SUMMARY")
        print("=" * 70)
        print(f"\n🎯 Quantitative Score: {final_state.get('quantitative_score', 'N/A')}/100")
        print(f"📊 Converted Score: {final_state.get('idea_score', 'N/A')}/10")
        
        if final_state.get("trend_score"):
            print(f"📈 Google Trends Score: {final_state.get('trend_score')}/100")
        
        competitors_count = len(final_state.get("competitors_data", {}).get("competitors", []))
        if competitors_count:
            print(f"🏢 Competitors Found: {competitors_count}")
        
        if final_state.get("market_research_results"):
            content = final_state["market_research_results"]
            lines = content.split('\n')
            print("\n🔍 Key Insight:")
            for line in lines[:10]:
                if any(word in line.lower() for word in ['problem', 'pain', 'gap', 'opportunity']):
                    print(f"   • {line[:100]}")
                    break
    
    # Save option
    print("\n" + "=" * 70)
    save = input("💾 Save full report to file? (y/n): ").strip().lower()
    
    if save == 'y':
        filename = f"startup_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write("=" * 70 + "\n")
            f.write("STARTUP ANALYSIS REPORT (Powered by Real Data)\n")
            f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("=" * 70 + "\n\n")
            f.write(f"IDEA: {idea_description}\n\n")
            f.write(f"QUANTITATIVE SCORE: {final_state.get('quantitative_score', 'N/A')}/100\n")
            f.write(f"CONVERTED SCORE: {final_state.get('idea_score', 'N/A')}/10\n\n")
            
            if final_state.get("trend_score"):
                f.write(f"GOOGLE TRENDS SCORE: {final_state.get('trend_score')}/100\n")
            
            competitors_count = len(final_state.get("competitors_data", {}).get("competitors", []))
            if competitors_count:
                f.write(f"COMPETITORS FOUND: {competitors_count}\n\n")
            
            if final_state.get("market_research_results"):
                f.write("=" * 70 + "\n")
                f.write("MARKET RESEARCH (Problems & Solutions)\n")
                f.write("=" * 70 + "\n")
                f.write(final_state["market_research_results"] + "\n\n")
            
            if final_state.get("astrology_advice"):
                f.write("=" * 70 + "\n")
                f.write("ASTROLOGY INSIGHTS\n")
                f.write("=" * 70 + "\n")
                f.write(final_state["astrology_advice"] + "\n")
        
        print(f"\n✅ Report saved to '{filename}'")
    
    print("\n" + "=" * 70)
    print("🚀 Good luck with your startup!")
    print("   Remember: Data-driven decisions lead to better outcomes")
    print("=" * 70 + "\n")

if __name__ == "__main__":
    main()


🚀 FOUNDER'S AI ASSISTANT (Powered by Real Data)

Data Sources:
  📊 Google Trends (SerpAPI) - Real market interest
  🔍 Tavily - Live competitor discovery
  🎯 Exa AI - Product Hunt & market validation
  🤖 DeepSeek V4 - Analysis & recommendations

I analyze your startup idea and provide:
  📊 1. Idea Evaluation (Data-driven score 0-100)
  📈 2. Market Research (Problems + Solutions)
  🌐 3. Domain Suggestions
  💰 4. VC Funding Analysis
  📋 5. CA/Compliance Advice
  💸 6. Tax Calculation
  📄 7. Sample Payslip
  ✨ 8. Astrology Motivation
  🤝 9. CRM Integration


🔧 API Status:
   DeepSeek: ✅
   SerpAPI:  ✅ (Google Trends)
   Tavily:   ✅ (Competitors)
   Exa AI:   ✅ (Market Validation)

💡 TELL ME ABOUT YOUR STARTUP IDEA
   (Include: problem you solve, target customer, what's unique)


✅ Analyzing: "Drone delivery system for small logistics like medicines and food..."

💰 FINANCIAL METRICS (Optional - improves accuracy)
   Press Enter to skip any question

   ⚠️ Invalid number. Using defaults.

🤖 